<a href="https://colab.research.google.com/github/Patrick190508/whaleshark-mafia-island/blob/main/notebooks/Model_v2_Ningaloo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model v2 — Daily whale shark encounter prediction, Ningaloo Reef
*(title and introduction to be written)*

Question: "Will today be a better-than-average day for its season?"
Data: `daily_ningaloo.csv` with wind, waves, their 3/7-day history, wind-noise index, lunar phase.

## 1. Data

### 1.1 Load the daily series and build the season table

In [2]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/Patrick190508/whaleshark-mafia-island/main/data/daily_ningaloo.csv"
d = pd.read_csv(URL, parse_dates=["date"])

PEAK = [4, 5, 6]                                   # Ningaloo season: April–June
d = d[d["month"].isin(PEAK)].copy()
d["sday"] = d.groupby("year").cumcount() + 1       # day of season 1..91
d["week"] = (d["sday"] - 1) // 7 + 1

# active seasons only (>= 20 sightings), as in the EDA
tot = d.groupby("year")["sightings"].sum()
active = tot[tot >= 20].index
d = d[d["year"].isin(active)].dropna(subset=["sst", "chl", "sss", "wind", "wave"]).copy()

# squared terms
d["sday2"] = d["sday"] ** 2
d["sst2"]  = d["sst"] ** 2

# target: better-than-average day for its own season
avg = d.groupby("year")["sightings"].transform("mean")
d["y"] = (d["sightings"] > avg).astype(int)

print("seasons:", sorted(d["year"].unique()))
print("rows:", len(d), "| good days:", d["y"].sum(), f"({d['y'].mean():.0%})")
print("columns:", list(d.columns))

seasons: [np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009)]
rows: 1092 | good days: 404 (37%)
columns: ['date', 'sst', 'chl', 'sss', 'sightings', 'doy', 'year', 'month', 'wind', 'wave', 'wind_3d', 'wind_7d', 'wave_3d', 'wave_7d', 'noise_wind', 'moon', 'moon_illum', 'sday', 'week', 'sday2', 'sst2', 'y']


### 1.2 Train / validation / test split by season

In [3]:
TRAIN = list(range(1998, 2005))
VAL   = [2005, 2006]
TEST  = [2007, 2008, 2009]

train = d[d["year"].isin(TRAIN)]
val   = d[d["year"].isin(VAL)]
test  = d[d["year"].isin(TEST)]

for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:11s} seasons {part['year'].min()}–{part['year'].max()} | "
          f"days {len(part):4d} | share of good days {part['y'].mean():.0%}")

train       seasons 1998–2004 | days  637 | share of good days 33%
validation  seasons 2005–2006 | days  182 | share of good days 43%
test        seasons 2007–2009 | days  273 | share of good days 42%


## 2. Metrics

### Why accuracy is not enough
Accuracy doesn't take into consideration the set imbalance. In our case with 63% of normal day the always-no model will have an accuracy of 63% and our best model had an accuracy of 66% in the testing set.

Than accuracy treat False positive and false negative at the same way but in the real world we may prefer a model with lower accuracy but with less false positive

### What are other metrics that we can use to analyze our model



*   Precision Score: To analyze the number of True positive (good days recognized by the model) over all the positive - TP/(TP+FP) - If high indicator of few False positives.
*   Recall Score of all better-than-avarage days: High recall = few better-than-avarage days lost
*   F1 Score: Avarage between Precision and recall scores - Usefull to check if the model is balanced between beeing precise and losing better-than-avarage days
*   ROC-AUC: a value that shows how well the model ranks the days: whether better-than-average days receive a higher probability than normal days. It does not depend on the threshold. 0.5 = random, 1 = perfect ranking.



### 2.2 Evaluation function
All metrics at one threshold, plus ROC-AUC (threshold-free).

In [4]:
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, confusion_matrix)

def evaluate(y_true, p, thr=0.5, label=""):
    """One row of metrics for a vector of probabilities p at threshold thr."""
    p = np.asarray(p, dtype=float)
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    row = {
        "model": label, "thr": thr,
        "accuracy":     accuracy_score(y_true, pred),
        "precision":    precision_score(y_true, pred, zero_division=0),
        "recall":       recall_score(y_true, pred, zero_division=0),
        "f1":           f1_score(y_true, pred, zero_division=0),
        "roc_auc":      roc_auc_score(y_true, p) if len(np.unique(p)) > 1 else 0.5,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    }
    return row

def metrics_table(rows):
    """Collect rows from evaluate() into one table."""
    t = pd.DataFrame(rows).set_index("model")
    return t.round(2)

## 3. Variables — do they separate good days from normal days?

### 3.1 Class tables on the training seasons
Share of better-than-average days per class of each variable, against the overall share.

*Reading:*

### 3.2 Correlation of each variable with the target
Point-biserial correlation (= Pearson between the 0/1 target and the variable) on the training seasons, sorted.

*Reading:*

## 4. Models

### 4.1 Features and scaling
Choose `FEATS` from section 3.

### 4.2 Weekly baseline

### 4.3 Logistic regression (linear) and with squared terms

### 4.4 Small neural network (Week 6)

### 4.5 Metrics on the validation set — all models, several thresholds
ROC-AUC to compare models; precision / recall to choose the threshold.

*Reading:*

### 4.6 Which variables does the model use? Coefficients, and whether their sign is stable between training and validation

*Reading:*

## 5. Test (used once)

*Reading:*

## 6. Outside the season — can the start and end of the season be predicted?

Same pipeline with target v1 (at least one sighting) on a wider window (March–July). Question: does the model predict the shoulder weeks better than the weekly baseline?

### 6.1 Wider season table and split

### 6.2 Baseline and models

### 6.3 Metrics on validation and test

*Reading:*

## 7. Conclusions
*(your text)*